# Nik Studio - animate

**You give it pictures and a song. It gives you one finished MP4.**

### Check first. It is free.

Colab charges for a GPU from the moment it connects, whatever you run on
it. So do the checking with no GPU at all:

1. **Runtime > Change runtime type > CPU**
2. Run **cell 2**. It finds your files, checks every one of them, and
   stops with `this cost nothing`.
3. Only once it says that: **Runtime > Change runtime type > L4 GPU**,
   run **cell 1**, then **cell 2** again.

A CPU runtime costs nothing, so a wrong folder or a picture that will
not open is found for free instead of on a meter.

Pick **L4**, not A100. The model is small and A100 spends units far
faster for no better result.

### Before you start

Make this folder in Google Drive and put your files in it:

```
My Drive / NikStudio / Input /
      Scene01.png
      Scene02.png
      Scene03.png
      song.mp3
```

Any names work - the pictures are used **in order**, so number them, and
any one audio file is taken as the song. `python tools\prepare.py --copy`
builds this folder for you and checks it before you ever open Colab.

The finished video comes back as:

```
My Drive / NikStudio / Output / Episode.mp4
```

### What it does

Every picture becomes a moving clip, the clips are cut to share the
length of the song exactly, and the song is laid over the top.

It reads the card and picks its own quality settings, so there is
nothing to tune.

It saves each clip to Drive as it finishes. If the session dies, run it
again - it picks up where it stopped instead of starting over. A picture
you replace is noticed and made again; the rest are not.

**Honest about one thing:** the mouth moves, but it is not lip-synced to
the words. Nothing free does real lip-sync yet.


In [ ]:
# ======================================================================
# CELL 1 of 2 - the packages.  Only needed once a GPU is turned on
# ======================================================================
#
# Skip this while you are still checking on a CPU runtime - cell 2 does
# the checking with what Colab already has.
#
# If Colab offers "RESTART SESSION" when this finishes, click it.
# Cell 2 depends on nothing in here, so a restart costs nothing.

!pip install -q "diffusers>=0.32" "transformers>=4.44" accelerate safetensors sentencepiece bitsandbytes imageio-ffmpeg librosa

print("Packages installed. Now run cell 2.")


In [ ]:
# ======================================================================
# CELL 2 of 2 - the whole thing
# ======================================================================
#
# Finds your pictures and your song in Drive, animates every picture,
# cuts the clips to share the song's length, lays the song over the top,
# and plays you the result.
#
# It depends on nothing above it, so running cells out of order or
# letting Colab restart the runtime cannot break it.

import gc
import json
import os
import re
import shutil
import subprocess
import time

from pathlib import Path

# Let the allocator grow its blocks instead of demanding one big
# contiguous run. On a card this full, fragmentation alone can be the
# difference between fitting and not.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch

from PIL import Image


# ======================================================================
# SETTINGS - the only part worth editing
# ======================================================================

DRIVE = "/content/drive/MyDrive"

FOLDER = DRIVE + "/NikStudio"

# Where the work goes when Drive will not connect. This session's own
# disk, which is emptied when the session ends.
LOCAL = "/content/NikStudio"

# What HAPPENS in the shot. Not what the picture shows - that is already
# there. One clear physical action beats three vague ones.
# One action for the character - and then a plain statement that
# everything else stays put.
#
# Whatever is not named drifts. Naming only the boy left the puppy, the
# kitten and the duckling free to melt, and they did, from about a
# second and a half in. And dropping "the camera does not move" let the
# whole frame push slowly in on its own.
#
# So: one movement, then who else is in the shot, then the camera.
PROMPT = (
    "The little boy sways gently from side to side, smiling. "
    "The puppy, the kitten and the duckling stay where they are, "
    "watching him. The background does not change. "
    "The camera does not move."
)

# A picture can have its own, by file name. Anything not listed here
# uses PROMPT above.
PROMPTS = {
    # "Scene02.png": "The boy splashes the water with both hands, ...",
}

# ------------------------------------------------- writing the scenes

# Put a script.txt in the Input folder - one scene a line - and the
# shots are made from your words instead of from pictures:
#
#     Nik waves at the puppy while the flowers sway around them
#     Nik runs down the garden path, the kitten chasing him
#
# Nothing is generated from a picture then, so nothing anchors the look.
# These two lines are what hold it together instead: the character is
# described the same way in every shot, and so is the style. Keep them
# exact and unchanging - a word altered between runs is a different boy.
CHARACTER = (
    "Nik, a chubby cheerful 3 year old Indian toddler boy, warm brown "
    "skin, soft wavy dark brown hair, very large expressive brown eyes "
    "with long lashes, round rosy cheeks, small button nose, wearing a "
    "bright blue t-shirt under yellow denim dungarees and blue canvas "
    "sneakers"
)

# The look. These words matter more than any other line in this cell.
#
# "children's cartoon" was in here, and that is precisely the phrase
# that fetched back flat ChuChu-TV shading - it is what most of the
# training data called itself. Naming the craft instead of the audience
# is what pulls it towards the render you actually want, and the flat
# look is pushed away in the negative prompt rather than merely not
# asked for.
STYLE = (
    "3D rendered CGI animation, Pixar and DreamWorks quality, "
    "physically based rendering, ray traced soft shadows, ambient "
    "occlusion, subsurface scattering on skin, glossy specular "
    "highlights, rounded three dimensional forms, deep background with "
    "real distance, volumetric sunlight, shallow depth of field, "
    "cinematic lighting, highly detailed"
)

# "static image, no movement" used to be in here, and it was the single
# worst line in this notebook: it tells the model to look UNLIKE the
# still it was given, which is the one thing it must not do. The face
# melted at two seconds and the scene was gone by four.
#
# Ask for the faults you do not want. Do not ask it to leave the picture.
NEGATIVE = (
    "worst quality, blurry, distorted, deformed face, melting face, "
    "morphing, warping, extra limbs, watermark, text, subtitles, "
    "camera zoom, camera pan, characters vanishing, "
    # The flat look has to be pushed away, not merely left unasked for.
    "flat shading, simple cartoon, low detail, cheap animation, "
    "mobile game graphics, plain empty background, stiff pose, "
    # The last clip came back as flat vector art with hard black
    # outlines. These are that look, named.
    "2d, vector art, flat colours, cel shading, black outlines, "
    "line art, clip art, storybook illustration, flash animation"
)

# Seconds per picture when there is no song to divide up.
FALLBACK_SECONDS = 5.0

FPS = 24

# Which model. Left blank, the machine decides: the 13B on a card with
# room for it, the 2B otherwise. Set it to "big" or "small" to insist.
FORCE_MODEL = ""

# Make the prompt actually count.
#
# The 13B we use by default is *distilled*, which is why it needs eight
# steps instead of fifty. The price of that is guidance_scale = 1.0, and
# at 1.0 there is no classifier-free guidance at all - which means the
# NEGATIVE prompt below is not read, and the positive one steers only
# weakly. That is why two attempts at fixing the art style by changing
# words changed almost nothing.
#
# True switches to the undistilled model, where guidance is real: the
# style words pull, and the negative prompt is obeyed. It costs about
# four times as long a clip, so try one shot with it before committing
# to a whole song.
USE_GUIDED_MODEL = False

# How long one generated clip is, before looping.
#
# This is the number that decides whether the video looks right, and
# shorter is safer. Everything in the picture drifts as the clip goes
# on, and the smallest things go first: over three seconds the
# butterflies had melted by one second and the animals' faces by about
# one and a half. Two seconds stays clean.
#
# A picture that has to be on screen longer is not given a longer clip.
# The clip is played forwards, then backwards, then forwards again, for
# as long as it is needed - a sway reads as continuous that way, and the
# model is never asked for more than it can do.
#
# Raise it for more movement in one go, at the cost of more drifting.
# The honest fix for a long song is more pictures, not longer clips.
CLIP_SECONDS = 2.0

# How long one shot holds the screen before the edit cuts away.
#
# This is the number that makes a video look made rather than assembled.
# Children's channels cut every two to four seconds and land every cut
# on the beat of the song; a shot that sits still for ten seconds reads
# as a slideshow however good the picture is.
#
# It also happens to be the fix for the model: cut before three seconds
# and a clip is never on screen long enough to drift.
SHOT_SECONDS = 2.8

# The bottom of every frame is thrown away, as a percentage of height.
# LTX 0.9.8 distilled stamps a line of garbled caption text there. No
# negative prompt removes it - with that model the negative prompt is
# not read at all - and 14 was not quite enough: it came back faintly on
# the next clip, sitting a little higher.
CROP_BOTTOM = 17

# The finished video. 1080p is what YouTube treats as HD.
OUTPUT_WIDTH, OUTPUT_HEIGHT = 1920, 1080

# Also write a vertical cut for Shorts.
MAKE_SHORT = True

# Make ONE clip from the first picture and stop.
#
# Worth doing before every real run, and certainly before a new
# character or a new prompt: two minutes of GPU tells you what the model
# does with your picture, instead of finding out eleven clips later.
# The song is ignored while this is on.
TEST_ONE_PICTURE = False


# The run stops when there are too few pictures for the song, because
# the clips would have to be slowed past the point of looking right
# and a paid GPU should not be spent finding that out. Set this True
# to go ahead anyway.
ALLOW_SLOW_CLIPS = False



# ======================================================================
# The card decides the quality, not you
# ======================================================================

# Not having one is fine here. Colab charges for a GPU from the moment
# it connects, whatever you run on it, so everything that does not need
# one is done first - on a CPU runtime, which costs nothing. Only when
# the files are known to be right is a GPU worth connecting.
HAS_GPU = torch.cuda.is_available()

if HAS_GPU:
    CARD = torch.cuda.get_device_name(0)
    VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9
else:
    # Assume the card the settings would be chosen for, so the advice
    # about picture counts is the advice you will actually get.
    CARD, VRAM = "none yet - checking your files first", 24.0

# Compute capability 8.0 (Ampere) or newer is where bfloat16 is real.
# Do not ask torch.cuda.is_bf16_supported() - it says True on a T4,
# because torch emulates bfloat16 in software rather than refusing, and
# that emulation is slower than it is worth.
MAJOR = torch.cuda.get_device_capability()[0] if HAS_GPU else 8

DTYPE = torch.bfloat16 if MAJOR >= 8 else torch.float16

# Ordinary RAM matters as much as the card here, and is the thing that
# actually killed the earlier attempts. The 9GB text encoder is unpacked
# in RAM before it ever reaches the GPU, so a big card on a small-RAM
# runtime still dies. Squeeze the text encoder to 8-bit whenever either
# one is short, not just when the card is.
try:
    import psutil

    RAM = psutil.virtual_memory().total / 1e9

except ImportError:
    # Colab ships psutil, but a notebook that dies on a missing helper
    # before it has even looked at your files is no use to anyone.
    import os

    RAM = (
        os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
    )

ROOMY_CARD = VRAM >= 20
ROOMY_RAM = RAM >= 20

# What the card decides is whether the 9GB text encoder has to be
# squeezed into 8-bit. It does NOT decide the video size: that is set by
# what the model was trained on, not by how much room there is.
QUANTISE = not (ROOMY_CARD and ROOMY_RAM)

# ---------------------------------------------------------- the model

# LTX comes in two sizes and the difference is not subtle.
#
# The 2B is the original. It runs anywhere and it drifts - the smallest
# things in a picture melt within a second or two.
#
# The 13B is distilled, which means it does in 8 steps what the 2B needs
# 50 for, and it takes image_cond_noise_scale - a setting the 2B's
# pipeline does not even have. At 0.0 no noise is added to your picture
# before it starts, which is precisely the thing that lets a clip wander
# away from it. It needs a 24GB card and room in ordinary RAM to be
# streamed through.
BIG = (VRAM >= 20 and RAM >= 30) if not FORCE_MODEL else FORCE_MODEL == "big"

if BIG and USE_GUIDED_MODEL:
    MODEL = "Lightricks/LTX-Video-0.9.7-dev"

    # Not distilled, so guidance works: the prompt pulls and the
    # negative prompt is read. Thirty steps rather than eight is the
    # price, and it is most of the wait.
    STEPS = 30
    TIMESTEPS = None
    GUIDANCE = 3.5

    WIDTH, HEIGHT = 960, 544

    QUANTISE = False

elif BIG:
    MODEL = "Lightricks/LTX-Video-0.9.8-13B-distilled"

    # Distilled: few steps, and the exact schedule it was distilled
    # for. These are not tuneable, they came with the model - and
    # guidance 1.0 means the negative prompt is not read at all.
    STEPS = 8
    TIMESTEPS = [1000, 993, 987, 981, 975, 909, 725, 0.03]
    GUIDANCE = 1.0

    WIDTH, HEIGHT = 960, 544

    # It offloads itself, layer by layer. An 8-bit text encoder pinned
    # to the card by device_map would only get in the way of that.
    QUANTISE = False

else:
    MODEL = "Lightricks/LTX-Video"

    STEPS = 50
    TIMESTEPS = None
    GUIDANCE = 3.0

    # This one was trained near 704x480 - about 338,000 pixels. Asking
    # for 1024x576 is 75% more, and it showed: the picture came apart
    # after two seconds. 768x448 is what it knows, in 16:9.
    WIDTH, HEIGHT = 768, 448

# Frames the model is asked for. (frames - 1) has to divide by 8, and
# the rounding goes to the nearest - rounding down turned a 3.0s clip
# into a 2.7s one.
MAX_FRAMES = max(25, round((CLIP_SECONDS * FPS - 1) / 8) * 8 + 1)

print(f"GPU         : {CARD}" + (f" ({VRAM:.0f}GB)" if HAS_GPU else ""))
print(f"System RAM  : {RAM:.0f}GB")
print(f"Precision   : {'bfloat16' if MAJOR >= 8 else 'float16'}"
      f"{', text encoder in 8-bit' if QUANTISE else ''}")
print(f"Model       : {MODEL.split('/')[-1]}"
      f" ({'13B' if BIG else '2B'}, {STEPS} steps)")
print(f"Guidance    : {GUIDANCE}"
      + ("  - the prompt pulls, the negative prompt is read"
         if GUIDANCE > 1.0 else
         "  - NO guidance: the negative prompt is ignored and the\n"
         "              style words steer only weakly. "
         "USE_GUIDED_MODEL = True to change that."))
print(f"Generated at: {WIDTH}x{HEIGHT}, "
      f"{MAX_FRAMES} frames ({MAX_FRAMES / FPS:.1f}s)")
print(f"Video out   : {OUTPUT_WIDTH}x{OUTPUT_HEIGHT}")


# ======================================================================
# Your files
# ======================================================================

try:
    from google.colab import files as colab_files
    from google.colab import drive as colab_drive

except ImportError:
    colab_files = colab_drive = None      # not in Colab


def mount_drive():
    """
    Connect Drive, and say plainly if it will not.

    Returns True when Drive is there. Not connecting is no longer fatal:
    the session's own disk works perfectly well, and asking someone to
    fight their browser settings before they can see a single clip is
    not a reasonable thing to do.
    """

    if colab_drive is None or Path(DRIVE).exists():
        return True

    try:
        colab_drive.mount("/content/drive")
        return True

    except Exception as trouble:

        # "credential propagation was unsuccessful" is the usual one,
        # and it is the browser refusing the sign-in popup - nothing to
        # do with Drive, the files, or this notebook.
        print(
            f"\n  Google Drive would not connect: {trouble}\n"
            "\n  That is the browser refusing the sign-in popup. If you "
            "want to use Drive:\n"
            "    1. Allow third-party cookies for "
            "colab.research.google.com - in Chrome, the\n"
            "       eye or padlock icon in the address bar > Cookies - "
            "and run this cell again.\n"
            "    2. An ordinary window, not Incognito or a guest "
            "profile.\n"
            "    3. Runtime > Disconnect and delete runtime, then "
            "reconnect.\n"
            "\n  Carrying on without it. Upload your files below and "
            "download the video at\n  the end - it works exactly the "
            "same, it just does not survive the session."
        )

        return False


ON_DRIVE = mount_drive()

root = Path(FOLDER) if ON_DRIVE else Path(LOCAL)

INPUT = root / "Input"
OUTPUT = root / "Output"
CLIPS = OUTPUT / "Clips"


def ask_for_files(folder):
    """Take the script, the song and any pictures straight from the PC."""

    folder.mkdir(parents=True, exist_ok=True)

    print("\n  Upload your script (.txt) and your song - and your "
          "pictures if you\n  are using pictures. You can pick them all "
          "at once.\n")

    for name in colab_files.upload():
        shutil.move(name, folder / name)

    print()


if not ON_DRIVE and colab_files is not None:

    if not INPUT.exists() or not any(INPUT.iterdir()):
        ask_for_files(INPUT)


def looks_like_input(folder):
    """A folder with pictures in it is a folder worth offering."""

    try:
        return any(
            item.suffix.lower() in (".png", ".jpg", ".jpeg", ".webp")
            for item in folder.iterdir()
        )
    except OSError:
        return False


def hunt_for_input():
    """
    Where the pictures actually are.

    "No such folder" is a dead end when the folder plainly exists on the
    other machine - Drive may be a different account, or the path may be
    a level off. Looking is more use than complaining, so anything named
    Input with pictures in it counts, and so does a NikStudio folder.
    """

    drive = Path(DRIVE)

    if not drive.exists():
        return []

    found = []

    for depth in ("*", "*/*", "*/*/*", "*/*/*/*"):

        for folder in drive.glob(depth):

            if not folder.is_dir():
                continue

            if folder.name.lower() in ("input", "nikstudio"):
                if looks_like_input(folder):
                    found.append(folder)

    return sorted(set(found))


if not INPUT.exists() and ON_DRIVE:

    print(f"\n  {INPUT} is not there. Looking for it ...")

    candidates = hunt_for_input()

    if len(candidates) == 1:

        INPUT = candidates[0]

        root = INPUT.parent
        OUTPUT = root / "Output"
        CLIPS = OUTPUT / "Clips"

        print(f"  Found your pictures in {INPUT} - using that.")

    elif candidates:

        raise SystemExit(
            f"No such folder: {INPUT}\n\n"
            "These have pictures in them - put the right one in FOLDER "
            "at the top of this cell\n(FOLDER is the folder ABOVE Input):"
            "\n\n"
            + "\n".join(f"    {folder}" for folder in candidates)
        )

    else:

        visible = sorted(
            item.name for item in Path(DRIVE).glob("*") if item.is_dir()
        ) if Path(DRIVE).exists() else []

        raise SystemExit(
            f"No such folder: {INPUT}\n\n"
            "Nothing with pictures in it was found anywhere in this "
            "Drive.\n\n"
            "Two things to check:\n"
            "  1. Colab is mounted on the same Google account your Drive "
            "folder is on.\n"
            "  2. Google Drive on your PC has finished uploading - a "
            "folder that only\n     exists locally is not there yet.\n\n"
            "The top level of the Drive Colab can see:\n\n"
            + ("\n".join(f"    {name}" for name in visible[:30])
               or "    (nothing)")
        )

CLIPS.mkdir(parents=True, exist_ok=True)

def natural_key(path):
    """
    Sort "Scene2" before "Scene10".

    Plain alphabetical order puts "10" before "2", which silently
    shuffles someone's scenes. Numbers in a name are compared as numbers.
    """

    return [
        int(part) if part.isdigit() else part.lower()
        for part in re.split(r"(\d+)", path.name)
    ]


PICTURES = sorted(
    (path for path in INPUT.iterdir()
     if path.suffix.lower() in (".png", ".jpg", ".jpeg", ".webp")),
    key=natural_key,
)

def is_lyrics(path):
    """A lyric sheet is any .txt whose name starts with "lyric"."""

    return path.stem.lower().startswith("lyric")


def lyrics_in(folder):
    """The words of the song, to put on screen. One line a line."""

    sheets = sorted(
        (item for item in folder.iterdir()
         if item.is_file() and item.suffix.lower() == ".txt"
         and is_lyrics(item)),
        key=natural_key,
    )

    if not sheets:
        return None, []

    lines = [
        line.strip()
        for line in sheets[0].read_text(encoding="utf-8").splitlines()
    ]

    return sheets[0], [l for l in lines if l and not l.startswith("#")]


def script_in(folder):
    """
    The script, whatever it ended up being called.

    Not `folder / "script.txt"`. Windows hides extensions, so renaming
    a file to "script.txt" in Explorer quietly produces script.txt.txt;
    Drive is case sensitive, so Script.txt is a different file; and
    people name things scenes.txt. There is no other reason for a .txt
    to be in here, so any of them is the script - preferring one that
    is at least called script.
    """

    texts = sorted(
        (item for item in folder.iterdir()
         if item.is_file() and item.suffix.lower() == ".txt"
         and not is_lyrics(item)),
        key=natural_key,
    )

    named = [t for t in texts if t.stem.lower().startswith("script")]

    return (named or texts)[0] if texts else None


SCRIPT = script_in(INPUT)

LYRIC_SHEET, LYRICS = lyrics_in(INPUT)

SONG = next(
    (
        path for path in sorted(INPUT.iterdir(), key=natural_key)
        if path.suffix.lower() in (".mp3", ".wav", ".m4a", ".aac", ".ogg")
    ),
    None,
)


def seconds_of(media):
    """How long an audio or video file runs, in seconds."""

    probe = subprocess.run(
        [
            "ffprobe", "-v", "error",
            "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1",
            str(media),
        ],
        capture_output=True,
        text=True,
    )

    try:
        return float(probe.stdout.strip())
    except ValueError:
        return 0.0


# ----------------------------------------------------------- the shots

# A shot is one clip: a name, the words the model is given, and the
# picture it starts from if there is one. Everything downstream works on
# these, so written scenes and pictures go through the same pipeline.

def from_script(path):

    lines = [
        line.strip()
        for line in path.read_text(encoding="utf-8").splitlines()
    ]

    # Blank lines space a script out; a # is how you park a scene
    # without deleting it.
    lines = [line for line in lines if line and not line.startswith("#")]

    return [
        {
            "name": f"Scene{number:02d}",
            "written": line,
            # The ACTION leads. It used to sit in the middle, after
            # sixty words describing the character, and a clip came
            # back of a boy standing still in an empty field - no
            # puppy, no flowers, none of what the line asked for.
            #
            # A video model weighs the front of a prompt hardest, and
            # this one weighs everything weakly, so burying the one
            # thing that is meant to happen behind a costume
            # description is throwing it away. Character second, style
            # last.
            #
            # Full stops between the three, or the action runs straight
            # into the description and they are read as one clause.
            "prompt": ". ".join(
                part.rstrip(" .,") for part in (line, CHARACTER, STYLE)
            ) + ".",
            "picture": None,
        }
        for number, line in enumerate(lines, start=1)
    ]


def from_pictures(paths):

    return [
        {
            "name": path.stem,
            "written": "",
            "prompt": PROMPTS.get(path.name, PROMPT),
            "picture": path,
        }
        for path in paths
    ]


if SCRIPT is not None:

    SHOTS = from_script(SCRIPT)

    if not SHOTS:
        raise SystemExit(
            f"{SCRIPT} has no scenes in it. Write one a line."
        )

    WRITTEN = True

    print(f"\nFrom       : {SCRIPT.name}, {len(SHOTS)} scene(s)")

    if PICTURES:
        print(f"             ({len(PICTURES)} picture(s) in the folder are "
              "being ignored - delete script.txt to use them instead)")

elif PICTURES:

    SHOTS = from_pictures(PICTURES)

    WRITTEN = False

    print(f"\nFrom       : {len(SHOTS)} picture(s)")

else:

    inside = sorted(item.name for item in INPUT.iterdir())

    raise SystemExit(
        f"Nothing to work from in {INPUT}.\n\n"
        "Put your pictures in there, or a script.txt with one scene a "
        "line.\n\n"
        "What is in there now:\n\n"
        + ("\n".join(f"    {name}" for name in inside[:40])
           or "    (nothing)")
    )


if TEST_ONE_PICTURE:

    SHOTS = SHOTS[:1]

    # Nothing is laid over a test clip. What comes back is the model's
    # work and nothing else.
    SONG = None

    SONG_SECONDS = 0.0

    SHARE = MAX_FRAMES / FPS

    print(f"TEST       : one clip only, {SHARE:.1f}s. The song is ignored.")

elif SONG:
    SONG_SECONDS = seconds_of(SONG)
    SHARE = SONG_SECONDS / len(SHOTS)
    print(f"Song       : {SONG.name} ({SONG_SECONDS:.1f}s)")

else:
    SONG_SECONDS = 0.0
    SHARE = FALLBACK_SECONDS
    print("Song       : none found - using "
          f"{FALLBACK_SECONDS:.0f}s per shot")

if LYRICS:
    print(f"Lyrics     : {LYRIC_SHEET.name}, {len(LYRICS)} line(s) "
          "- they go on screen")

print(f"Shots      : {len(SHOTS)} "
      f"({', '.join(shot['name'] for shot in SHOTS)})")

# Not "each holds 7.8s" - the edit cuts every SHOT_SECONDS, so nothing
# is on screen for that long. What that number really is, is each
# shot's share of the song.
print(f"Each gets  : {SHARE:.1f}s of the song, cut into pieces")

# ======================================================================
# Everything that can go wrong, found before the GPU is touched
# ======================================================================
#
# Loading the model takes minutes and a paid GPU is charged for them, so
# nothing here is left to be discovered halfway through the run.

problems = []

print()

for shot in SHOTS:

    if shot["picture"] is None:
        print(f"  {shot['name']:<10} {shot['written'][:56]}")
        continue

    picture_file = shot["picture"]

    try:
        with Image.open(picture_file) as check:
            check.verify()

        with Image.open(picture_file) as check:
            shape = check.size

    except Exception:
        problems.append(
            f"{picture_file.name} will not open. Re-save it as a PNG."
        )
        continue

    print(f"  {picture_file.name:<28} {shape[0]}x{shape[1]}")

if SONG and not SONG_SECONDS:
    problems.append(
        f"{SONG.name} cannot be read. Try a plain MP3 or WAV."
    )

# The edit cuts every SHOT_SECONDS, so a long song is no longer a
# problem of holding one picture too long. It is a question of variety:
# with more shots than clips, each clip comes back more than once.
#
# Twice or three times is how television works - you return to a setup.
# Eight times over two minutes is the same three seconds again and
# again, and no camera move hides that.
SHOTS_IN_EDIT = max(1, round((SONG_SECONDS or SHARE * len(SHOTS))
                             / SHOT_SECONDS))

TIMES_EACH = SHOTS_IN_EDIT / len(SHOTS)

if TIMES_EACH > 8:

    enough = max(1, round(SHOTS_IN_EDIT / 4))

    complaint = (
        f"{len(SHOTS)} shot(s) against {SHOTS_IN_EDIT} cuts means each "
        f"clip comes back {TIMES_EACH:.0f} times.\n"
        f"       The edit cannot hide that. Use about {enough} shots for "
        f"a {SONG_SECONDS:.0f}s song.\n\n"
        f"       Only trying one out? Set TEST_ONE_PICTURE = True at the "
        f"top - it makes\n       one clip, ignores the song, and takes "
        f"about two minutes. Or ALLOW_SLOW_CLIPS = True\n"
        f"       to go ahead as things stand."
    )

    if ALLOW_SLOW_CLIPS:
        print(f"\n  Warning: {complaint}")
    else:
        problems.append(complaint)

elif TIMES_EACH > 4:
    print(f"\n  Note: {SHOTS_IN_EDIT} cuts from {len(SHOTS)} clips, so "
          f"each comes back about {TIMES_EACH:.0f} times.\n"
          f"        Watchable - the camera move differs each time - but "
          f"more shots would be better.")

elif TIMES_EACH > 1.2:
    print(f"\n  {SHOTS_IN_EDIT} cuts from {len(SHOTS)} clips: each is "
          f"seen about {TIMES_EACH:.0f} times, from a\n  different "
          f"camera move each time.")

if problems:

    raise SystemExit(
        "\n\nStopping before the GPU is used:\n\n"
        + "\n".join(f"  -  {problem}" for problem in problems)
        + "\n\nFix these and run this cell again. Nothing has been "
          "charged for."
    )

print("\n  Everything checks out.")

if not HAS_GPU:

    raise SystemExit(
        "\n\nYour files are ready, and this cost nothing - there was no "
        "GPU running.\n\n"
        "Now turn one on and do the work:\n\n"
        "  1. Runtime > Change runtime type > L4 GPU > Save\n"
        "  2. Run cell 1 (the packages), then run this cell again.\n\n"
        f"It will make {len(SHOTS)} clip(s). Nothing else needs "
        "checking."
    )


# ======================================================================
# The model.  Kept between runs - loading it is most of the wait
# ======================================================================

if "pipe" not in globals():

    if BIG:
        from diffusers import LTXConditionPipeline as Pipeline
    else:
        from diffusers import LTXImageToVideoPipeline as Pipeline

    print("\nLoading the model. A few minutes the first time"
          + (" - the 13B is a big download." if BIG else "."))

    started = time.time()

    parts = {}

    if QUANTISE:

        from transformers import BitsAndBytesConfig, T5EncoderModel

        # 9GB of text encoder will not fit through a small card's RAM at
        # full size. In 8-bit it goes straight to the GPU at about 4.7GB.
        parts["text_encoder"] = T5EncoderModel.from_pretrained(
            MODEL,
            subfolder="text_encoder",
            quantization_config=BitsAndBytesConfig(load_in_8bit=True),
            device_map="auto",
        )

    if BIG:

        from diffusers import AutoModel
        from diffusers.hooks import apply_group_offloading

        # 13B in bfloat16 is 26GB against a 24GB card, and moving whole
        # components on and off does not help when one component is the
        # thing that does not fit. Two steps make it fit:
        #
        #   fp8 storage      - the weights sit in half the space and are
        #                      converted back a layer at a time as they
        #                      are used. Needs an Ada card or newer.
        #   leaf offloading  - only the small piece being computed is on
        #                      the card at all; the rest waits in RAM.
        transformer = AutoModel.from_pretrained(
            MODEL, subfolder="transformer", torch_dtype=DTYPE,
        )

        transformer.enable_layerwise_casting(
            storage_dtype=torch.float8_e4m3fn, compute_dtype=DTYPE,
        )

        pipe = Pipeline.from_pretrained(
            MODEL, transformer=transformer, torch_dtype=DTYPE, **parts
        )

        onload = torch.device("cuda")
        offload = torch.device("cpu")

        pipe.transformer.enable_group_offload(
            onload_device=onload,
            offload_device=offload,
            offload_type="leaf_level",
            use_stream=True,
        )

        apply_group_offloading(
            pipe.text_encoder,
            onload_device=onload,
            offload_type="block_level",
            num_blocks_per_group=2,
        )

        apply_group_offloading(
            pipe.vae, onload_device=onload, offload_type="leaf_level",
        )

    else:

        pipe = Pipeline.from_pretrained(MODEL, torch_dtype=DTYPE, **parts)

        if QUANTISE:
            # The text encoder is already on the card; move what is left.
            pipe.transformer.to("cuda")
            pipe.vae.to("cuda")
        else:
            pipe.to("cuda")

    # The 13B has placed itself: nothing sits on the card until it is
    # needed, so there is nothing to move here.

    # Decoding every frame in one piece is what runs a card out of
    # memory. Tiling decodes it in patches instead.
    pipe.vae.enable_tiling()

    print(f"Model ready in {time.time() - started:.0f}s "
          f"({torch.cuda.memory_allocated() / 1e9:.1f}GB on the card).")

else:
    print("\nModel already loaded - reusing it.")


# ======================================================================
# One picture -> one clip
# ======================================================================

def fitted(picture):
    """Cover the frame and crop the overflow, rather than squash."""

    scale = max(WIDTH / picture.width, HEIGHT / picture.height)

    picture = picture.resize(
        (round(picture.width * scale), round(picture.height * scale)),
        Image.LANCZOS,
    )

    left = (picture.width - WIDTH) // 2
    top = (picture.height - HEIGHT) // 2

    return picture.crop((left, top, left + WIDTH, top + HEIGHT))


def animate(shot, clip_file, seconds):
    """
    One shot -> one clip of exactly `seconds`.

    A shot either starts from a picture or from a line of your script.
    Everything after that is the same either way.

    The model is always asked for the same short clip, however long the
    shot has to be on screen. Asking it for a long one is what made the
    face melt. The clip is then played forwards, backwards and forwards
    again until the time is filled: a sway reads as continuous that way,
    and the seam is at the moment the movement turns around, where it is
    least visible.

    Returns (frames, loops) - what was generated, and how many times the
    forward-and-back pair had to run.
    """

    image = (
        fitted(Image.open(shot["picture"]).convert("RGB"))
        if shot["picture"] is not None else None
    )

    prompt = shot["prompt"]

    def run(count):

        asked = dict(
            prompt=prompt,
            negative_prompt=NEGATIVE,
            width=WIDTH,
            height=HEIGHT,
            num_frames=count,
            frame_rate=FPS,
            num_inference_steps=STEPS,
            guidance_scale=GUIDANCE,
            # Stated rather than left to a default. The tokenizer warns
            # about 128 because that is its own configured length; the
            # pipeline allows 256, and a full prompt here is about 180.
            # Left implicit, a future default could silently cut the
            # style words off the end - where they sit.
            max_sequence_length=256,
            generator=torch.Generator("cpu").manual_seed(42),
        )

        # Written scenes send no picture at all: the pipeline makes the
        # whole thing from the words. Same model, same everything else.
        if image is not None:
            asked["image"] = image

        if BIG:
            asked.update(
                timesteps=TIMESTEPS,
                decode_timestep=0.05,
                decode_noise_scale=0.025,
            )

            if image is not None:
                # No noise on the conditioning picture. This is the
                # setting that keeps the clip on the picture it was
                # given, and the 2B pipeline has no equivalent.
                asked["image_cond_noise_scale"] = 0.0

        return pipe(**asked).frames[0]

    frames = MAX_FRAMES

    try:
        video = run(frames)

    except torch.cuda.OutOfMemoryError:

        gc.collect()
        torch.cuda.empty_cache()

        frames = max(25, ((frames // 2 - 1) // 8) * 8 + 1)

        print(f"    card ran out of room - shorter clip ({frames} frames)")

        try:
            video = run(frames)

        except torch.cuda.OutOfMemoryError as full:

            # A shorter clip does not help when it is the model that
            # will not fit. Say what to do rather than repeat 34 frames
            # of traceback.
            raise SystemExit(
                "The card is out of memory even on the shortest clip.\n\n"
                + (
                    "You are on the 13B model. Either lower CLIP_SECONDS "
                    "and the size at the top,\nor set FORCE_MODEL = "
                    '"small" to use the 2B, which fits anywhere.\n\n'
                    if BIG else
                    "Lower CLIP_SECONDS at the top of this cell, or use "
                    "a bigger GPU.\n\n"
                )
                + "Runtime > Restart session first - a half loaded model "
                "is still holding the card.\n\n"
                "Clips already finished are safe in Output/Clips and will "
                "not be made again."
            ) from full

    from diffusers.utils import export_to_video

    from diffusers.utils import export_to_video

    raw = clip_file.with_name(clip_file.stem + "_raw.png.mp4")

    export_to_video(video, str(raw), fps=FPS)

    # ------------------------------------------------- the model's mark

    # LTX 0.9.8 distilled was trained on captioned video and stamps a
    # line of garbled text along the bottom of everything it makes. No
    # negative prompt shifts it. Cutting the band off is what works, and
    # the edit re-frames afterwards so nothing looks short of picture.
    keep = HEIGHT - (round(HEIGHT * CROP_BOTTOM / 100) // 2) * 2

    subprocess.run(
        [
            "ffmpeg", "-y", "-loglevel", "error",
            "-i", str(raw),
            "-vf", f"crop={WIDTH}:{keep}:0:0",
            "-r", str(FPS),
            "-c:v", "libx264", "-pix_fmt", "yuv420p",
            "-preset", "veryfast", "-crf", "16",
            str(clip_file),
        ],
        check=True,
    )

    raw.unlink(missing_ok=True)

    return frames


# ======================================================================
# Every picture
# ======================================================================

# A clip is reused only when the thing it was made from has not moved.
# Skipping by file name alone is what would quietly leave you with a clip
# of last week's Scene02 after you replaced the picture.
STAMPS = CLIPS / "made.json"

try:
    stamps = json.loads(STAMPS.read_text(encoding="utf-8"))
except Exception:
    stamps = {}


def stamp_for(shot):

    # What the clip was generated from - and nothing about the song or
    # the edit. Swapping the song used to count as a change and threw
    # away every clip, when the clips would have been identical.
    made_from = {
        "prompt": shot["prompt"],
        "size": f"{WIDTH}x{HEIGHT}",
        "frames": MAX_FRAMES,
        "model": MODEL,
    }

    if shot["picture"] is not None:

        facts = shot["picture"].stat()

        made_from.update(
            picture=shot["picture"].name,
            bytes=facts.st_size,
            modified=int(facts.st_mtime),
        )

    return made_from


made = []

for number, shot in enumerate(SHOTS, start=1):

    clip_file = CLIPS / f"{shot['name']}.mp4"

    label = f"[{number}/{len(SHOTS)}] {shot['name']}"

    wanted = stamp_for(shot)

    finished = clip_file.exists() and clip_file.stat().st_size > 0

    if finished and stamps.get(clip_file.name) == wanted:
        print(f"{label}: already made, skipping")
        made.append(clip_file)
        continue

    if finished:
        print(f"{label}: changed since last time - making it again")

    print(f"{label}: animating ...")

    started = time.time()

    frames = animate(shot, clip_file, SHARE)

    print(f"    done in {(time.time() - started) / 60:.1f} min "
          f"({frames / FPS:.1f}s of movement)")

    # Written after every clip, not at the end: a session that dies has
    # to leave behind an honest record of what is really finished.
    stamps[clip_file.name] = wanted

    STAMPS.write_text(json.dumps(stamps, indent=1), encoding="utf-8")

    made.append(clip_file)


# ======================================================================
# The edit
# ======================================================================
#
# This is the part that decides whether the video looks made or
# assembled, and it costs no GPU at all.
#
# A children's channel does not put one picture on screen for eleven
# seconds. It cuts every two to four seconds, it lands every cut on the
# beat of the song, and it comes back to the same setup later from a
# different angle. Eleven clips become forty shots that way, and the
# song carries the rhythm of the edit.
#
# It is also, by luck, exactly what this model needs: nothing is on
# screen long enough to drift.

def beats_of(song):
    """
    Where the beats fall, in seconds.

    Falls back to an even grid when the song cannot be analysed - a
    steady edit is still better than one long held shot.
    """

    if song is None:
        return []

    try:
        import librosa

        samples, rate = librosa.load(str(song), sr=22050, mono=True)

        _, frames = librosa.beat.beat_track(y=samples, sr=rate)

        times = librosa.frames_to_time(frames, sr=rate)

        return [float(t) for t in times]

    except Exception as trouble:

        print(f"  (could not find the beat - {type(trouble).__name__} - "
              "cutting on an even grid instead)")

        return []


def cuts_for(beats, seconds):
    """
    Where the shots change, in seconds, from 0 to the end.

    Beats are gathered up until a shot has run about SHOT_SECONDS, so
    every cut lands on one. Without beats it is an even grid.

    Returns (points, on_beat), because saying "cut on the beat" when it
    was really a stopwatch is the sort of thing you then believe.
    """

    beats = [t for t in beats if 0 < t < seconds]

    if not beats:
        count = max(1, round(seconds / SHOT_SECONDS))
        step = seconds / count
        return [step * n for n in range(count + 1)], False

    points = [0.0]

    for beat in beats:
        if beat - points[-1] >= SHOT_SECONDS:
            points.append(beat)

    # A last shot shorter than half a shot is a flash; give its time to
    # the one before instead.
    if seconds - points[-1] < SHOT_SECONDS / 2 and len(points) > 1:
        points.pop()

    points.append(seconds)

    return points, True


# ------------------------------------------------------------- camera

# Every shot moves. A still frame from a video model reads as a freeze,
# and a slow push or drift is what the eye expects from animation. They
# cycle, so returning to the same clip does not look like a repeat.
MOVES = ("push_in", "pan_right", "hold", "pull_out", "pan_left", "push_in_up")


def pulse_at(beats, frames, strength=0.05, fall=0.18):
    """
    A little push on every beat, as a zoompan expression.

    This is what "movement with the song" actually means. The cuts
    already land on the beat, but between cuts nothing was answering
    the music - and this model gives so little movement of its own
    that the picture sat there.

    Each beat adds a small bump that falls away over `fall` seconds, so
    the frame breathes in time rather than drifting past the rhythm.

    Measured against a still picture: the camera move alone comes out
    at 1.81, and with this it is 4.26 - more than double, with the
    peaks landing on the beats. 0.08 was jumpy.
    """

    if not beats:
        return ""

    terms = []

    for beat in beats:
        # `on` is the output frame, so on/FPS is the second we are at.
        terms.append(
            f"{strength}*max(0\,1-abs(on/{FPS}-{beat:.3f})/{fall})"
        )

    return "+" + "+".join(terms)


def move_filter(move, frames, beats=()):
    """
    A slow camera move across the clip, ending up at the output size.

    The clip is scaled well past the output first: zoompan works in
    whole source pixels, so moving across a frame-sized picture makes
    the movement visibly step.
    """

    # Only a little past the output. The clip itself is 960 wide, so
    # scaling to 4K before moving across it buys nothing that is really
    # there and costs minutes per shot - and on Colab minutes are money.
    big_w = round(OUTPUT_WIDTH * 1.2 / 2) * 2
    big_h = round(OUTPUT_HEIGHT * 1.2 / 2) * 2

    last = max(1, frames - 1)

    # 1.16 was a lurch and 1.06 was nothing: with it the whole video
    # measured a third of the movement it had before, because the model
    # itself barely moves and the camera was carrying all of it. This
    # is the middle, and the beat pulse below is what actually makes it
    # feel animated.
    zoom = 1.0 if move == "hold" else 1.11

    centre_x = "iw/2-(iw/zoom/2)"
    centre_y = "ih/2-(ih/zoom/2)"
    span_x = "(iw-iw/zoom)"
    span_y = "(ih-ih/zoom)"

    growing = f"1+{zoom - 1:.6f}*on/{last}"
    shrinking = f"{zoom:.6f}-{zoom - 1:.6f}*on/{last}"
    held = f"{zoom:.6f}"

    moves = {
        # A held shot is still scaled and cropped, just not moved.
        "hold":       (f"{1.0:.6f}", centre_x, centre_y),
        "push_in":    (growing, centre_x, centre_y),
        "pull_out":   (shrinking, centre_x, centre_y),
        "pan_right":  (held, f"{span_x}*on/{last}", centre_y),
        "pan_left":   (held, f"{span_x}*(1-on/{last})", centre_y),
        "push_in_up": (growing, centre_x, f"{span_y}*(1-on/{last})"),
    }

    expression, x, y = moves.get(move, moves["push_in"])

    # The beat rides on top of whatever the camera was already doing.
    expression = f"({expression}){pulse_at(beats, frames)}"

    return (
        # bicubic, not lanczos. The clip is 960 wide, so there is no
        # detail for a sharper filter to find - it just costs twice the
        # time, and on Colab time is money.
        f"scale={big_w}:{big_h}:force_original_aspect_ratio=increase:"
        f"flags=bicubic,"
        f"crop={big_w}:{big_h},"
        f"zoompan=z='{expression}':x='{x}':y='{y}'"
        f":d=1:s={OUTPUT_WIDTH}x{OUTPUT_HEIGHT}:fps={FPS},"
        f"format=yuv420p"
    )


def bounce_of(clip_file):
    """
    The clip forwards then backwards, so it can run for ever.

    A frame is dropped at each end of the reversed half: the first is
    the one the forward half just finished on and the last is the one
    the loop is about to start on again, and holding either for two
    frames shows as a hitch.
    """

    # Kept in the Edit folder, which is emptied every run. Keeping it
    # beside the clip meant a regenerated clip quietly kept the loop
    # made from the old one - and the edit would still be showing last
    # week's picture.
    bounced = EDIT / f"{clip_file.stem}_loop.mp4"

    frames = MAX_FRAMES

    subprocess.run(
        [
            "ffmpeg", "-y", "-loglevel", "error",
            "-i", str(clip_file),
            "-filter_complex",
            "[0:v]split[fwd][back];"
            f"[back]reverse,trim=start_frame=1:end_frame={frames - 1},"
            "setpts=PTS-STARTPTS[rev];"
            "[fwd][rev]concat=n=2:v=1[out]",
            "-map", "[out]",
            "-r", str(FPS),
            "-c:v", "libx264", "-pix_fmt", "yuv420p",
            "-preset", "veryfast", "-crf", "16",
            str(bounced),
        ],
        check=True,
    )

    return bounced


def cut_shot(source, target, seconds, move, beats=()):
    """One shot of the finished video: looped to length, and moving."""

    frames = max(2, round(seconds * FPS))

    subprocess.run(
        [
            "ffmpeg", "-y", "-loglevel", "error",
            "-stream_loop", "-1", "-i", str(source),
            "-t", f"{seconds:.3f}",
            "-vf", move_filter(move, frames, beats),
            "-r", str(FPS),
            # A working file, encoded fast and near-losslessly. The one
            # encode that matters is the last one.
            "-c:v", "libx264", "-pix_fmt", "yuv420p",
            "-preset", "ultrafast", "-crf", "14",
            str(target),
        ],
        check=True,
    )


# ======================================================================
# Cut it together
# ======================================================================

EDIT = OUTPUT / "Edit"

if EDIT.exists():
    shutil.rmtree(EDIT)

EDIT.mkdir(parents=True)

length = SONG_SECONDS or (SHARE * len(made))

BEATS = beats_of(SONG)

points, on_beat = cuts_for(BEATS, length)

print(f"\nEditing: {len(points) - 1} shot(s) across {length:.1f}s"
      + (", cut on the beat" if on_beat else ", cut on an even grid"))

loops = [bounce_of(clip) for clip in made]

pieces = []

for number in range(len(points) - 1):

    start, finish = points[number], points[number + 1]

    # Which shot this cut belongs to, by where it falls in the song.
    #
    # Not the cut number cycled round the shots: that would run scene
    # one, two, three ... eleven and then jump back to scene one, which
    # throws away the order your script is in. Each shot gets its own
    # stretch of the song instead, and is cut up several times within
    # it - a different camera move each time.
    which = min(len(loops) - 1, int(start / length * len(loops)))

    source = loops[which]

    move = MOVES[number % len(MOVES)]

    piece = EDIT / f"{number:03d}.mp4"

    # The beats inside this shot, counted from the shot's own start,
    # because the filter sees a clip that begins at zero.
    inside = [b - start for b in BEATS if start < b < finish]

    cut_shot(source, piece, finish - start, move, inside)

    pieces.append(piece)

print(f"         {len(pieces)} cuts from {len(loops)} clip(s), "
      f"in the order you wrote them")


# ======================================================================
# The words of the song, on screen
# ======================================================================
#
# This is the one thing that makes a rhyme video feel like it is *of*
# the song rather than merely over it - children sing along with it, and
# it is what every channel in this corner of YouTube does.
#
# It costs no GPU at all. Put a lyrics.txt in the Input folder, one line
# a line, and the lines are spread across the song on its beats.

def font_for(text):
    """
    A font that can actually draw these words.

    Devanagari in a font that has no Devanagari is a row of empty
    boxes, and it is better to say so than to render that.
    """

    wanted = ["Noto Sans Devanagari", "Lohit Devanagari", "Mukta",
              "Noto Sans", "DejaVu Sans", "Liberation Sans"]

    try:
        listed = subprocess.run(
            ["fc-list", "--format", "%{family}\n"],
            capture_output=True, text=True, timeout=30,
        ).stdout

    except Exception:
        listed = ""

    families = {name.strip() for line in listed.splitlines()
                for name in line.split(",")}

    devanagari = any("\u0900" <= ch <= "\u097f" for ch in text)

    for name in wanted:

        if name not in families:
            continue

        if devanagari and "Devanagari" not in name and "Noto Sans" != name:
            continue

        return name, True

    if devanagari:
        print("  ! The lyrics are in Devanagari and no font here can "
              "draw it - they would\n    come out as empty boxes, so "
              "they are being left off. In cell 1 add:\n"
              "        !apt-get -qq install -y fonts-indic")
        return None, False

    return "DejaVu Sans", True


def spread_over(lines, beats, seconds):
    """
    When each line of the song is on screen.

    Split on the beat, the same way the cuts are, so the words change
    with the music instead of on a stopwatch. Without beats it is an
    even share.
    """

    if not lines:
        return []

    usable = [t for t in beats if 0 < t < seconds]

    if len(usable) < len(lines):
        step = seconds / len(lines)
        edges = [step * n for n in range(len(lines) + 1)]

    else:
        # The beat nearest each line's fair share of the song.
        edges = [0.0]
        for n in range(1, len(lines)):
            ideal = seconds * n / len(lines)
            edges.append(min(usable, key=lambda t: abs(t - ideal)))
        edges.append(seconds)

    return list(zip(lines, edges[:-1], edges[1:]))


def timecode(seconds):
    """0:00:01.50, which is what ASS wants."""

    hours, rest = divmod(max(0.0, seconds), 3600)
    minutes, rest = divmod(rest, 60)

    return f"{int(hours)}:{int(minutes):02d}:{rest:05.2f}"


def write_lyrics(timed, target, width, height, font):
    """An ASS subtitle file, styled for a children's video."""

    size = round(height * 0.068)
    margin = round(height * 0.075)

    escaped = str(target)

    target.write_text(
        "[Script Info]\n"
        "ScriptType: v4.00+\n"
        f"PlayResX: {width}\n"
        f"PlayResY: {height}\n"
        "WrapStyle: 0\n"
        "\n"
        "[V4+ Styles]\n"
        "Format: Name, Fontname, Fontsize, PrimaryColour, OutlineColour, "
        "BackColour, Bold, BorderStyle, Outline, Shadow, Alignment, "
        "MarginL, MarginR, MarginV, Encoding\n"
        # White with a thick dark outline: readable over a bright sky
        # and over dark grass alike, without a box behind it.
        f"Style: Sing,{font},{size},&H00FFFFFF,&H00303030,&H80000000,"
        f"-1,1,{max(2, round(size * 0.07))},2,2,"
        f"{round(width * 0.06)},{round(width * 0.06)},{margin},1\n"
        "\n"
        "[Events]\n"
        "Format: Layer, Start, End, Style, Name, MarginL, MarginR, "
        "MarginV, Effect, Text\n"
        + "".join(
            f"Dialogue: 0,{timecode(start)},{timecode(end)},Sing,,0,0,0,,"
            "{\\fad(120,120)}" + line.replace("\n", " ") + "\n"
            for line, start, end in timed
        ),
        encoding="utf-8",
    )

    return target


# ======================================================================
# The finished file
# ======================================================================

listing = OUTPUT / "shots.txt"

listing.write_text(
    "".join(f"file '{piece}'\n" for piece in pieces),
    encoding="utf-8",
)

FINAL = OUTPUT / "Episode.mp4"

# The wide video and the vertical one each need the words drawn at their
# own size, so the master is made without them and each burns its own.
MASTER = OUTPUT / "_master.mp4" if LYRICS else FINAL

command = [
    "ffmpeg", "-y", "-loglevel", "error",
    "-f", "concat", "-safe", "0", "-i", str(listing),
]

if SONG:
    command += [
        "-i", str(SONG),
        # YouTube turns everything down to about -14 LUFS. A quiet
        # upload stays quiet next to a channel that mastered theirs.
        "-af", "loudnorm=I=-14:TP=-1.5:LRA=11",
        "-c:a", "aac", "-b:a", "192k", "-ar", "48000", "-ac", "2",
        "-shortest",
    ]

command += [
    "-c:v", "libx264", "-profile:v", "high", "-level", "4.0",
    "-pix_fmt", "yuv420p",
    "-preset", "medium", "-crf", "18",
    "-r", str(FPS),
    "-movflags", "+faststart",
    str(MASTER),
]

subprocess.run(command, check=True)

listing.unlink(missing_ok=True)


# The vertical cut.
#
# This was a centre crop, then a fit-with-blurred-background, and the
# fit was worse: a 16:9 frame inside 9:16 fills under a third of the
# height, so most of a Short was a huge blurred face with a thin strip
# of video in the middle.
#
# So: crop, but from a frame whose camera is now calm. The character is
# in the middle of every shot because that is how the clips are framed,
# so the middle is what to keep.
# No output label: the caller adds one, because the lyrics have to be
# chained on after the crop.
VERTICAL = (
    "[0:v]crop=trunc(ih*9/16/2)*2:ih,"
    "scale=1080:1920:flags=bicubic"
)


def lyric_filter(width, height, seconds):
    """
    The filter that draws the words, or "" if they cannot be drawn.

    The path goes inside a filter argument, where a colon separates
    options - so it has to be escaped or a Windows drive letter reads
    as two options.
    """

    font, usable = font_for(" ".join(LYRICS))

    if not usable:
        return ""

    sheet = write_lyrics(
        spread_over(LYRICS, BEATS, seconds),
        OUTPUT / f"lyrics_{width}x{height}.ass",
        width, height, font,
    )

    escaped = str(sheet).replace("\\", "/").replace(":", "\\:")

    return f"subtitles='{escaped}'"


def encode(source, target, chain, seconds=None, complex_chain=False):
    """One re-encode with a filter chain, kept in one place."""

    command = ["ffmpeg", "-y", "-loglevel", "error", "-i", str(source)]

    if seconds:
        command += ["-t", str(seconds)]

    if chain:
        command += (["-filter_complex", chain, "-map", "[out]", "-map", "0:a?"]
                    if complex_chain else ["-vf", chain])

    command += [
        "-c:v", "libx264", "-profile:v", "high", "-pix_fmt", "yuv420p",
        "-preset", "medium", "-crf", "18",
        "-c:a", "aac", "-b:a", "192k",
        "-movflags", "+faststart",
        str(target),
    ]

    subprocess.run(command, check=True)


if LYRICS:

    words = lyric_filter(OUTPUT_WIDTH, OUTPUT_HEIGHT, seconds_of(MASTER))

    if words:
        encode(MASTER, FINAL, words)
        print("\nLyrics  : drawn on, changing on the beat")
    else:
        shutil.copyfile(MASTER, FINAL)

print(f"\nFinished: {FINAL}")
print(f"Length  : {seconds_of(FINAL):.1f}s at {OUTPUT_WIDTH}x{OUTPUT_HEIGHT}")

# ------------------------------------------------------------- Shorts

if MAKE_SHORT and not TEST_ONE_PICTURE:

    SHORT = OUTPUT / "Episode_Short.mp4"

    # Built from the master, not from the finished wide video: the words
    # have to be drawn at the vertical size, and burning them twice
    # would show the wide ones shrunk underneath.
    vertical = VERTICAL

    if LYRICS:
        tall_words = lyric_filter(1080, 1920, min(55.0, seconds_of(MASTER)))

        if tall_words:
            vertical = f"{VERTICAL}[fitted];[fitted]{tall_words}[out]"
        else:
            vertical = f"{VERTICAL}[out]"
    else:
        vertical = f"{VERTICAL}[out]"

    encode(MASTER, SHORT, vertical, seconds=55, complex_chain=True)

    print(f"Shorts  : {SHORT} ({seconds_of(SHORT):.0f}s, 1080x1920)")

if not ON_DRIVE and colab_files is not None:

    # Nothing here survives the session, so hand the file over rather
    # than leave it somewhere that is about to be deleted.
    print("\nDownloading it to your PC - the session keeps no copy.")

    colab_files.download(str(FINAL))

    if MAKE_SHORT and not TEST_ONE_PICTURE:
        colab_files.download(str(SHORT))

if LYRICS and MASTER != FINAL:
    MASTER.unlink(missing_ok=True)

from IPython.display import Video, display

display(Video(str(FINAL), embed=True, width=720))

# ----------------------------------------------------------------------
# The file is in Drive, so it is already on your PC if Drive syncs there.
#
# Want one shot to do something else? Change its line in script.txt, or
# add the picture to PROMPTS at the top, and run this cell again. Only
# what changed is generated afresh - the rest are kept, and the edit is
# rebuilt every time because cutting costs nothing.
# ----------------------------------------------------------------------
